# 05 — Grey Wolf Optimizer (GWO) Feature Selection

Binary GWO searches for a compact feature mask. The implementation keeps the best-so-far wolf instead of accidentally returning only the best wolf from the last iteration.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Grey Wolf Optimizer

In [2]:
# -------------------------------------------------
# Binary Grey Wolf Optimizer (BGWO)
# -------------------------------------------------

import numpy as np


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def run_bgwo(
    obj_func,
    n_features,
    pop_size=30,
    iterations=50,
):

    # Initialize wolves
    positions = np.random.randint(0, 2, (pop_size, n_features))

    fitness = np.array([obj_func(w) for w in positions])

    order = np.argsort(fitness)

    alpha = positions[order[0]].copy()
    beta = positions[order[1]].copy()
    delta = positions[order[2]].copy()

    # Preserve the best solution found across every iteration.
    best_position = alpha.copy()
    best_score = float(fitness[order[0]])

    convergence = []

    for t in range(iterations):

        a = 2 - 2 * (t / iterations)

        for i in range(pop_size):

            new_position = np.zeros(n_features)

            for j in range(n_features):

                # -------- Alpha --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A1 = 2 * a * r1 - a
                C1 = 2 * r2

                D_alpha = abs(C1 * alpha[j] - positions[i, j])
                X1 = alpha[j] - A1 * D_alpha

                # -------- Beta --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A2 = 2 * a * r1 - a
                C2 = 2 * r2

                D_beta = abs(C2 * beta[j] - positions[i, j])
                X2 = beta[j] - A2 * D_beta

                # -------- Delta --------
                r1 = np.random.rand()
                r2 = np.random.rand()

                A3 = 2 * a * r1 - a
                C3 = 2 * r2

                D_delta = abs(C3 * delta[j] - positions[i, j])
                X3 = delta[j] - A3 * D_delta

                X = (X1 + X2 + X3) / 3

                prob = sigmoid(X)

                new_position[j] = 1 if np.random.rand() < prob else 0

            # Prevent empty feature subset
            if new_position.sum() == 0:
                new_position[np.random.randint(n_features)] = 1

            positions[i] = new_position

        # Evaluate
        fitness = np.array([obj_func(w) for w in positions])

        order = np.argsort(fitness)

        alpha = positions[order[0]].copy()
        beta = positions[order[1]].copy()
        delta = positions[order[2]].copy()

        current_best_score = float(fitness[order[0]])

        if current_best_score < best_score:
            best_score = current_best_score
            best_position = alpha.copy()

        convergence.append(best_score)

    return best_position, best_score, convergence

## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def gwo_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bgwo(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
    )


gwo_results = run_feature_selector("GWO", gwo_runner)
gwo_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'svm', 1)


Saved: ('breast', 'svm', 2)


Saved: ('breast', 'svm', 3)


Saved: ('breast', 'svm', 4)


...

Saved: ('heart', 'xgboost', 1)


Saved: ('heart', 'xgboost', 2)


Saved: ('heart', 'xgboost', 3)


Saved: ('heart', 'xgboost', 4)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
25,heart,xgboost,GWO,0,0.126370,0.793478,0.840426,0.774510,0.806122,0.890842,20,21.610136,0.094910,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Femal...",results/final/artifacts/heart__xgboost__gwo__s...,results/final/artifacts/heart__xgboost__gwo__s...
26,heart,xgboost,GWO,1,0.119789,0.788043,0.838710,0.764706,0.800000,0.889527,17,21.464556,0.092044,"[""chol"", ""oldpeak"", ""ca"", ""sex_Female"", ""cp_as...",results/final/artifacts/heart__xgboost__gwo__s...,results/final/artifacts/heart__xgboost__gwo__s...
27,heart,xgboost,GWO,2,0.123970,0.760870,0.815217,0.735294,0.773196,0.867647,14,21.534484,0.095207,"[""chol"", ""thalch"", ""oldpeak"", ""cp_asymptomatic...",results/final/artifacts/heart__xgboost__gwo__s...,results/final/artifacts/heart__xgboost__gwo__s...
28,heart,xgboost,GWO,3,0.120589,0.782609,0.836957,0.754902,0.793814,0.893352,19,21.599243,0.094177,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Male""...",results/final/artifacts/heart__xgboost__gwo__s...,results/final/artifacts/heart__xgboost__gwo__s...
29,heart,xgboost,GWO,4,0.121789,0.777174,0.821053,0.764706,0.791878,0.888092,22,21.591073,0.097373,"[""age"", ""trestbps"", ""chol"", ""thalch"", ""oldpeak...",results/final/artifacts/heart__xgboost__gwo__s...,results/final/artifacts/heart__xgboost__gwo__s...
